# Analysis: README tables and figures from `evaluation/summary.csv`

Run with `make report` (it executes this notebook). Every number below is read from `evaluation/summary.csv`
(written by `make exp1` ... `make exp4`) or `configs/base.yaml` (`published:`); nothing is typed by hand.
The tables are written between the `<!-- BEGIN:x -->` / `<!-- END:x -->` markers of README.md and evaluation/README.md.

In [1]:
import re

import matplotlib.pyplot as plt
import pandas as pd

from src.config import load_config, repo_path

S = pd.read_csv(repo_path("evaluation/summary.csv")).set_index(["experiment", "model"])
PUB, CLASSES = load_config()["published"]["attnsleep"], ["W", "N1", "N2", "N3", "REM"]
pct = lambda v: f"{100 * v:.1f}"
ci = lambda r, k: f"{pct(r[k])} [{pct(r[k + '_lo'])}, {pct(r[k + '_hi'])}]" if k != "kappa" else f"{r[k]:.3f} [{r[k + '_lo']:.3f}, {r[k + '_hi']:.3f}]"
S.reset_index()[["experiment", "model", "n_seeds", "acc", "mf1", "kappa", "f1_N1"]].round(3)

,experiment,model,n_seeds,acc,mf1,kappa,f1_N1
0,exp1_main,lgbm,1,0.824,0.764,0.760,0.417
1,exp1_main,attnsleep,1,0.814,0.758,0.747,0.406
2,exp1_main,tinysleepnet,3,0.820,0.772,0.758,0.477
3,exp1_main,seqctx_transformer,3,0.834,0.787,0.776,0.505
4,exp2_leakage,lgbm_subject,1,0.824,0.764,0.760,0.417
5,exp2_leakage,lgbm_epoch,1,0.920,0.885,0.890,0.683
6,exp3_context,lgbm_nocontext,1,0.794,0.720,0.720,0.296
7,exp3_context,lgbm,1,0.824,0.764,0.760,0.417
8,exp3_context,tinysleepnet,3,0.820,0.772,0.758,0.477
9,exp4_missingness,keep,1,0.824,0.764,0.760,0.417


## Main result (exp1) vs published AttnSleep

In [2]:
rows = ["| Model | Acc % | Macro-F1 % | Kappa | " + " | ".join(f"F1 {c}" for c in CLASSES) + " | Seeds |", "|---" * 10 + "|"]
inside = []
for m, r in S.loc["exp1_main"].iterrows():
    seeds = r.note + (f" (acc range {pct(r.acc_seed_min)}-{pct(r.acc_seed_max)})" if r.n_seeds > 1 else "")
    rows.append(f"| {r.label} | {ci(r, 'acc')} | {ci(r, 'mf1')} | {ci(r, 'kappa')} | " + " | ".join(pct(r[f'f1_{c}']) for c in CLASSES) + f" | {seeds} |")
    pos = lambda k, v: "inside" if r[k + "_lo"] <= v <= r[k + "_hi"] else ("above" if v > r[k + "_hi"] else "below")
    inside.append(f"- {m}: acc {pos('acc', PUB['acc'] / 100)}, macro-F1 {pos('mf1', PUB['mf1'] / 100)}, kappa {pos('kappa', PUB['kappa'])}")
rows.append(f"| *AttnSleep, published (Eldele et al. 2021, Table V)*¹ | {PUB['acc']} | {PUB['mf1']} | {PUB['kappa']} | "
            + " | ".join(str(PUB["f1"][c]) for c in CLASSES) + " | n/a |")
main = "\n".join(rows + ["", "Is the published value inside our 95% CI? (inside = \"matches\")", ""] + inside)
print(main)

| Model | Acc % | Macro-F1 % | Kappa | F1 W | F1 N1 | F1 N2 | F1 N3 | F1 REM | Seeds |
|---|---|---|---|---|---|---|---|---|---|
| LightGBM, YASA-style features + neighbouring-epoch context | 82.4 [79.7, 84.9] | 76.4 [73.0, 79.3] | 0.760 [0.723, 0.794] | 85.9 | 41.7 | 87.2 | 87.9 | 79.3 | 1 seed |
| AttnSleep reproduction (epoch chosen on validation subjects) | 81.4 [78.7, 83.8] | 75.8 [73.0, 78.2] | 0.747 [0.711, 0.780] | 86.1 | 40.6 | 87.3 | 88.7 | 76.1 | 1 seed |
| TinySleepNet (CNN + LSTM, 20-epoch windows) | 82.0 [77.6, 85.4] | 77.2 [72.5, 80.7] | 0.758 [0.699, 0.802] | 86.2 | 47.7 | 85.6 | 82.9 | 83.4 | mean of 3 seeds (acc range 81.8-82.5) |
| seqctx_transformer (TinySleepNet CNN + Transformer; own design) | 83.4 [79.6, 86.4] | 78.7 [74.8, 81.8] | 0.776 [0.724, 0.816] | 87.5 | 50.5 | 86.3 | 83.3 | 85.7 | mean of 3 seeds; chosen as our best deep model after seeing test results (selection bias) (acc range 82.5-84.3) |
| *AttnSleep, published (Eldele et al. 2021, Table V)*¹ | 84.4 

## Leakage (exp2), context (exp3), missingness (exp4)

In [3]:
def small(exp, keys=("acc", "mf1", "kappa", "f1_N1")):
    head = ["| | Epochs | Acc % | Macro-F1 % | Kappa | F1 N1 % |", "|---|---|---|---|---|---|"]
    return "\n".join(head + [f"| {r.label} | {r.n_epochs:,} | " + " | ".join(ci(r, k) for k in keys) + " |" for _, r in S.loc[exp].iterrows()])

lk = S.loc["exp2_leakage"]
leak = small("exp2_leakage") + f"\n| **Inflation (epoch - subject)** | | +{pct(lk.acc.lgbm_epoch - lk.acc.lgbm_subject)} pp | " \
       f"+{pct(lk.mf1.lgbm_epoch - lk.mf1.lgbm_subject)} pp | +{lk.kappa.lgbm_epoch - lk.kappa.lgbm_subject:.3f} | +{pct(lk.f1_N1.lgbm_epoch - lk.f1_N1.lgbm_subject)} pp |"
blocks = dict(main=main, leakage=leak, context=small("exp3_context"), missingness=small("exp4_missingness"))
for name in ("leakage", "context", "missingness"):
    print(blocks[name], "\n")

| | Epochs | Acc % | Macro-F1 % | Kappa | F1 N1 % |
|---|---|---|---|---|---|
| Subject-wise 20-fold CV (honest) | 42,308 | 82.4 [79.7, 84.9] | 76.4 [73.0, 79.3] | 0.760 [0.723, 0.794] | 41.7 [31.9, 49.1] |
| Epoch-wise random 20-fold split (leaky) | 42,308 | 92.0 [91.3, 92.7] | 88.5 [87.5, 89.3] | 0.890 [0.880, 0.899] | 68.3 [64.1, 71.5] |
| **Inflation (epoch - subject)** | | +9.6 pp | +12.1 pp | +0.130 | +26.6 pp | 

| | Epochs | Acc % | Macro-F1 % | Kappa | F1 N1 % |
|---|---|---|---|---|---|
| LightGBM, current-epoch features only (no context) | 42,308 | 79.4 [76.4, 82.1] | 72.0 [68.9, 74.9] | 0.720 [0.680, 0.755] | 29.6 [20.7, 37.4] |
| LightGBM + context (7.5-min, 2-min and +-2-epoch windows, time of night) | 42,308 | 82.4 [79.7, 84.9] | 76.4 [73.0, 79.3] | 0.760 [0.723, 0.794] | 41.7 [31.9, 49.1] |
| TinySleepNet (LSTM over 20-epoch windows), mean of 3 seeds | 42,308 | 82.0 [77.6, 85.4] | 77.2 [72.5, 80.7] | 0.758 [0.699, 0.802] | 47.7 [39.5, 54.4] | 

| | Epochs | Acc % | Macr

In [4]:
def fill(path, blocks):
    text = repo_path(path).read_text()
    for name, body in blocks.items():
        if f"<!-- BEGIN:{name} -->" in text:
            text = re.sub(rf"(<!-- BEGIN:{name} -->\n).*?(<!-- END:{name} -->)", lambda m: m.group(1) + body + "\n" + m.group(2), text, flags=re.S)
    repo_path(path).write_text(text)

fill("README.md", blocks)
fill("evaluation/README.md", blocks)

## Figure: macro-F1 with 95% subject-bootstrap CIs

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharex=True)
for ax, (exp, title) in zip(axes, [("exp2_leakage", "Leakage"), ("exp3_context", "Temporal context"), ("exp4_missingness", "Missingness")]):
    d = S.loc[exp].iloc[::-1]
    ax.errorbar(100 * d.mf1, range(len(d)), xerr=[100 * (d.mf1 - d.mf1_lo), 100 * (d.mf1_hi - d.mf1)], fmt="o", capsize=3)
    ax.set_yticks(range(len(d)), [l if len(l) < 38 else l[:36] + "..." for l in d.label], fontsize=8)
    ax.set_title(title)
    ax.set_xlabel("Macro-F1 % (95% CI over subjects)")
fig.tight_layout()
(repo_path("evaluation/figures")).mkdir(exist_ok=True)
fig.savefig(repo_path("evaluation/figures/experiments.png"), dpi=130)
plt.close(fig)